# demo_safety

> Phase 9: bad agents (liars, free-riders, saboteurs), defences (trust, reputation, graduated sanctions with spot checks), liar detection, and the pre-registered test H3.

Spec: `specs/multiagent/phase9/` (design-low-level section 7). Artifacts under `results/multiagent/safety/{quick,h3,full}/` (gitignored): `runs.csv`, `metrics.json`, `report.md` and figures. Normal tests run every grid in quick mode (pilot seeds, budgets / 10; a pipeline check); `--cell-timeout 0 --flags slow` runs H3 once on seeds 300-329; `--cell-timeout 0 --flags sweeps` runs the exploratory attack, defence and monitoring grids on seeds 400-409.

In [ ]:
import os
import subprocess
import time

import matplotlib.pyplot as plt

from explRL.bench_grid import confirm, runGrid, writeCSV
from explRL.safety_figures import safetyFigures
from explRL.safety_grid import (PREREG_H3, THETA, attackGrid, defenceGrid, h3Grid, monitorGrid, preregHashH3,
                                runSafetyCell)
from explRL.safety_report import report, summaries
from explRL.safety_runs import lossRows
from explRL.train import demo_artifact_dir, save_demo_figure, save_demo_metrics, save_demo_report

DN = 'multiagent/safety'
WORKERS = min(8, os.cpu_count() or 1)
DEVIATIONS = 'none'


def gitCommit():
    try:
        return subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], capture_output=True, text=True).stdout.strip() or 'unknown'
    except OSError:
        return 'unknown'


def run(mode, cells, seeds, h3seeds=None):
    "Run cells, analyse, and write runs.csv, figures, metrics.json and report.md under results/multiagent/safety/<mode>/."
    t0 = time.time()
    rows = runGrid(cells, WORKERS, run=runSafetyCell)
    wall = time.time() - t0
    conf = confirm(lossRows(rows), PREREG_H3, seeds=h3seeds) if any(r['study'] == 'h3' for r in rows) else None
    tabs = summaries(rows)
    figs = safetyFigures(tabs)
    writeCSV(rows, demo_artifact_dir(mode, DN) / 'runs.csv')
    for name, f in figs.items():
        save_demo_figure(name, fig=f, mode=mode, demo_name=DN)
    plt.close('all')
    meta = dict(mode=mode, cells=len(cells), workers=WORKERS, wallSeconds=wall, seeds=seeds, preregHash=preregHashH3(),
                gitCommit=gitCommit(), prereg=PREREG_H3, deviations=DEVIATIONS, theta=THETA, figures=list(figs))
    jtabs = {n: {'/'.join(str(x) for x in k): v for k, v in t.items()} for n, t in tabs.items()}
    save_demo_metrics({**meta, 'h3': conf, 'tables': jtabs}, mode=mode, demo_name=DN)
    path = save_demo_report(report(meta, conf, tabs), mode=mode, demo_name=DN)
    print(f'{mode}: {len(rows)} rows in {wall:.0f} s ->', path)
    if conf:
        c = conf['H3']; print('H3', c['verdict'], round(c['mean'], 4), [round(x, 4) for x in c['ci']], 'p', round(c['p'], 4))
    return rows

## Quick: every grid on pilot seeds (pipeline check)

In [ ]:
quick = run('quick', attackGrid('quick') + defenceGrid('quick') + monitorGrid('quick') + h3Grid('quick'),
            'pilot 1000-1001 (budgets / 10)', h3seeds=(1000, 1001))

## H3, confirmatory (`--flags slow`; run once on the frozen pre-registration)

In [ ]:
#| slow
h3 = run('h3', h3Grid('full'), 'confirmatory 300-329')

## Exploratory grids (`--flags sweeps`)

In [ ]:
#| sweeps
full = run('full', attackGrid('full') + defenceGrid('full') + monitorGrid('full'), 'exploratory 400-409')